In [1]:
import json
import re
from pathlib import Path
from tqdm import tqdm

# 🔁 CHANGE THIS PER COLAB
SPLIT_NAME = "split_1"

DATA_DIR = Path("/content/drive/MyDrive/data_v2")
SPLITS_DIR = Path("/content/drive/MyDrive/data_v2/splits")

SPLIT_FILE = SPLITS_DIR / f"{SPLIT_NAME}.txt"
DONE_FILE = SPLITS_DIR / f"{SPLIT_NAME}_done.txt"

In [2]:
!pip install -q bitsandbytes accelerate transformers

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
def is_non_empty(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, list):
        return len(value) > 0
    return False


def get_category(description, transcription_en):
    if is_non_empty(description):
        return 0
    elif is_non_empty(transcription_en):
        return 1
    else:
        return 2

In [4]:
def remove_timestamps(lines):
    cleaned = []
    for line in lines:
        line = re.sub(r"\[\d{2}:\d{2}:\d{2}\.\d+\]", "", line)
        cleaned.append(line.strip())
    return cleaned


def smart_slice(text, max_chars=4500):
    n = len(text)
    if n <= max_chars:
        return text

    part = max_chars // 3
    return text[:part] + text[n//2 - part//2 : n//2 + part//2] + text[-part:]


FOOD_HINTS = ["ingredients", "cook", "recipe", "fry", "boil", "taste"]
NEWS_HINTS = ["news", "report", "update", "breaking", "government"]

def extract_signal_lines(lines, max_lines=10):
    selected = []
    for line in lines:
        l = line.lower()
        if any(k in l for k in FOOD_HINTS + NEWS_HINTS):
            selected.append(line)
        if len(selected) >= max_lines:
            break
    return " ".join(selected)

In [5]:
def build_cat0_input(title, description):
    return f"""
[TYPE: DESCRIPTION]

[TITLE]
{title}

[DESCRIPTION]
{description}
"""


def build_cat1_input(title, transcript_lines):
    cleaned = remove_timestamps(transcript_lines)
    joined = " ".join(cleaned)

    sliced = smart_slice(joined)
    signals = extract_signal_lines(cleaned)

    return f"""
[TYPE: TRANSCRIPT]

[TITLE]
{title}

[TRANSCRIPT_SNIPPET]
{sliced}

[IMPORTANT_LINES]
{signals}
"""


def build_cat2_input(title):
    return f"""
[TYPE: TITLE_ONLY]

[TITLE]
{title}
"""

In [6]:
VALID = ["food", "news", "other", "unpredictable"]

def build_prompt(text):
    return f"""
You are a strict classifier.

Classify the YouTube video into ONE of:
food, news, other, unpredictable

Rules:
- Output ONLY one word
- No explanation

{text}

Answer:
"""


def run_batch(prompts):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")

    outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


import re

def parse_output(output):
    output = output.lower().strip()

    for line in reversed(output.split("\n")):
        line = line.strip()

        # remove punctuation
        line = re.sub(r"[^a-z]", "", line)

        for v in VALID:
            if line == v:
                return v

    return "unpredictable"

In [7]:
def load_split_file():
    with open(SPLIT_FILE, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    return lines


def load_done_set():
    if not DONE_FILE.exists():
        return set()

    with open(DONE_FILE, "r") as f:
        return set(l.strip() for l in f if l.strip())


lines = load_split_file()
done_set = load_done_set()

pending_lines = [l for l in lines if l not in done_set]

print(f"Total: {len(lines)} | Done: {len(done_set)} | Pending: {len(pending_lines)}")

Total: 587 | Done: 88 | Pending: 499


In [8]:
def prepare_batches(lines, batch_size=8):
    batches = []

    current = []

    for line in lines:
        path, cat = line.rsplit(" ", 1)
        cat = int(cat)

        current.append((path, cat))

        if len(current) >= batch_size:
            batches.append(current)
            current = []

    if current:
        batches.append(current)

    return batches


batches = prepare_batches(pending_lines, batch_size=8)

In [ ]:
with open(DONE_FILE, "a") as done_f:

    for batch in tqdm(batches, desc="Processing Batches"):

        prompts = []
        meta = []

        for path, cat in batch:

            try:
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                title = data.get("metadata", {}).get("title", "")
                description = data.get("metadata", {}).get("description", "")
                transcription_en = data.get("transcription_english", [])

                if cat == 0:
                    text = build_cat0_input(title, description)
                elif cat == 1:
                    text = build_cat1_input(title, transcription_en)
                else:
                    text = build_cat2_input(title)

                prompt = build_prompt(text)

                prompts.append(prompt)
                meta.append((path, cat, data))

            except Exception as e:
                print(f"Error reading {path}: {e}")

        # LLM call
        outputs = run_batch(prompts)

        # Process outputs
        for out, (path, cat, data) in zip(outputs, meta):

            label = parse_output(out)

            # update JSON
            data["video_type"] = label

            try:
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                # write to done file
                done_f.write(f"{path} {cat}\n")
                done_f.flush()

            except Exception as e:
                print(f"Error writing {path}: {e}")

Processing Batches:  62%|██████▏   | 39/63 [10:02<06:02, 15.09s/it]